# 07 - Validación extractor + scaler + API

Este notebook valida que `feature_extractor_top25.py`, el modelo, el scaler y la API est?n alineados.

In [1]:
import os
import sys
from pathlib import Path

sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

BASE_DIR = Path.cwd()
if not (BASE_DIR / "feature_extractor_top25.py").exists():
    BASE_DIR = Path(r"C:\phishing-url-classifier\phishing-url-ai")

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print("BASE_DIR:", BASE_DIR)


BASE_DIR: c:\phishing-url-classifier\phishing-url-ai


In [2]:
import importlib
import pandas as pd
import sklearn

import feature_extractor_top25 as extractor
extractor = importlib.reload(extractor)

extract_top25_features = extractor.extract_top25_features
scale_top25_features = extractor.scale_top25_features
load_feature_names = extractor.load_feature_names
load_continuous_features = extractor.load_continuous_features
load_model = extractor.load_model


def predict_from_raw_features(url, raw_features):
    scaled_features = scale_top25_features(raw_features)
    prediction = int(model.predict(scaled_features)[0])

    probability = None
    if hasattr(model, "predict_proba"):
        classes = list(model.classes_)
        probabilities = model.predict_proba(scaled_features)[0]
        probability = float(probabilities[classes.index(1)]) if 1 in classes else float(max(probabilities))

    return {
        "url": extractor.normalize_url(url),
        "prediction": prediction,
        "label_extractor": "phishing" if prediction == 1 else "legitimate",
        "label_api": "phishing" if prediction == 1 else "no-phishing",
        "probability": probability,
        "raw_features": raw_features,
        "scaled_features": scaled_features,
    }

MODEL_FEATURES = load_feature_names()
CONTINUOUS_FEATURES = load_continuous_features()
model = load_model()

print("Extractor:", extractor.__file__)
print("scikit-learn:", sklearn.__version__)
print("Cantidad de features:", len(MODEL_FEATURES))
print("Cantidad de features continuas:", len(CONTINUOUS_FEATURES))
print("Orden coincide:", list(model.feature_names_in_) == MODEL_FEATURES)
print("Clases:", list(model.classes_))


Extractor: c:\phishing-url-classifier\phishing-url-ai\feature_extractor_top25.py
scikit-learn: 1.7.2
Cantidad de features: 25
Cantidad de features continuas: 20
Orden coincide: True
Clases: [0, 1]


## Configuración de prueba

Usa `USE_NETWORK_FEATURES = True` para comparar contra la API.

In [3]:
URL_TEST = "https://www.google.com"
USE_NETWORK_FEATURES = True

# Importante: se extraen features una sola vez, igual que en la API.
raw_features = extract_top25_features(
    URL_TEST,
    use_network_features=USE_NETWORK_FEATURES,
)
result = predict_from_raw_features(URL_TEST, raw_features)
scaled_features = result["scaled_features"]

print("URL:", result["url"])
print("Prediction:", result["prediction"])
print("Label extractor:", result["label_extractor"])
print("Label API:", result["label_api"])
print("Probability phishing:", result["probability"])
print("Raw shape:", raw_features.shape)
print("Scaled shape:", scaled_features.shape)
print("Orden raw correcto:", list(raw_features.columns) == MODEL_FEATURES)
print("Orden scaled correcto:", list(scaled_features.columns) == MODEL_FEATURES)
print("Nulos raw/scaled:", int(raw_features.isna().sum().sum()), int(scaled_features.isna().sum().sum()))


URL: https://www.google.com
Prediction: 0
Label extractor: legitimate
Label API: no-phishing
Probability phishing: 0.20333333333333334
Raw shape: (1, 25)
Scaled shape: (1, 25)
Orden raw correcto: True
Orden scaled correcto: True
Nulos raw/scaled: 0 0


## Comparación local con varias URLs

In [4]:
urls = [
    "https://www.google.com",
    "https://www.pcfactory.cl",
    "http://login-banco-seguro.xyz",
]

rows = []
local_results = {}

for url in urls:
    # Una sola extracci?n por URL.
    raw = extract_top25_features(url, use_network_features=USE_NETWORK_FEATURES)
    item = predict_from_raw_features(url, raw)
    local_results[url] = item
    rows.append({
        "url": item["url"],
        "label_extractor": item["label_extractor"],
        "label_api": item["label_api"],
        "prediction": item["prediction"],
        "probability_phishing": item["probability"],
    })

pd.DataFrame(rows)


,url,label_extractor,label_api,prediction,probability_phishing
0,https://www.google.com,legitimate,no-phishing,0,0.203333
1,https://www.pcfactory.cl,legitimate,no-phishing,0,0.050000
2,http://login-banco-seguro.xyz,phishing,phishing,1,0.796667


## Comparación contra API Flask

Ejecuta esta celda solo si la API est? levantada en `http://127.0.0.1:5000`.

In [10]:
import os
import requests
from dotenv import load_dotenv

load_dotenv(Path(r"C:\phishing-url-classifier\phishing-url-api") / ".env")
API_BASE_URL = "http://127.0.0.1:5000"
API_KEY = os.getenv("API_KEY")

urls_compare = [
    "https://www.google.com",
    "https://www.pcfactory.cl",
    "http://login-banco-seguro.xyz",
]

api_rows = []
diff_rows = []

for url in urls_compare:
    # Una sola extracci?n local por URL en esta celda.
    local_raw_features = extract_top25_features(url, use_network_features=USE_NETWORK_FEATURES)
    local_item = predict_from_raw_features(url, local_raw_features)
    local_features = local_raw_features.iloc[0].to_dict()

    response = requests.post(
        f"{API_BASE_URL}/predict",
        json={"url": url},
        headers={"X-API-KEY": API_KEY},
        timeout=60,
    )
    data = response.json()
    api_features = data.get("explanation", {}).get("features", {})

    feature_diffs = []
    feature_diffs_values = []

    for feature in MODEL_FEATURES:
        local_value = float(local_features.get(feature, 0))
        api_value = float(api_features.get(feature, 0))
        difference = local_value - api_value

        if abs(difference) > 1e-12:
            feature_diffs.append(feature)
            feature_diffs_values.append(
                f"{feature}: local={local_value}, api={api_value}, diff={difference}"
            )
            diff_rows.append({
                "url": url,
                "feature": feature,
                "local_value": local_value,
                "api_value": api_value,
                "difference": difference,
            })

    api_rows.append({
        "url": url,
        "status_code": response.status_code,
        "local_label": local_item["label_api"],
        "api_label": data.get("label"),
        "local_probability": round(float(local_item["probability"]), 6),
        "api_probability": data.get("probability"),
        "feature_diffs": ", ".join(feature_diffs),
        "feature_diffs_values": " | ".join(feature_diffs_values),
    })

comparison_df = pd.DataFrame(api_rows)
diff_df = pd.DataFrame(diff_rows)

display(comparison_df)

if diff_df.empty:
    print("No hay diferencias de features entre local y API.")
else:
    display(diff_df)


,url,status_code,local_label,api_label,local_probability,api_probability,feature_diffs,feature_diffs_values
0,https://www.google.com,200,no-phishing,no-phishing,0.203333,0.203333,,
1,https://www.pcfactory.cl,200,no-phishing,no-phishing,0.050000,0.060000,"domain_age, domain_registration_length","domain_age: local=9646.0, api=0.0, diff=9646.0..."
2,http://login-banco-seguro.xyz,200,phishing,phishing,0.796667,0.796667,,


,url,feature,local_value,api_value,difference
0,https://www.pcfactory.cl,domain_age,9646.0,0.0,9646.0
1,https://www.pcfactory.cl,domain_registration_length,605.0,0.0,605.0


## Diagn?stico WHOIS, Google Index y HTML

Esta celda prueba las mismas funciones de red que usa el extractor. Si WHOIS falla, ver?s `domain_age = 0` y `domain_registration_length = 0`.

In [6]:
diagnostic_urls = [
    "https://www.google.com",
    "https://www.pcfactory.cl",
    "http://login-banco-seguro.xyz",
]

diagnostic_rows = []
google_query_rows = []

for url in diagnostic_urls:
    try:
        html, final_url, history, status_code = extractor.fetch_html(url)
        html_ok = html is not None
    except Exception as error:
        final_url = extractor.normalize_url(url)
        history = []
        status_code = None
        html_ok = False
        html_error = str(error)
    else:
        html_error = ""

    whois_info = extractor.get_domain_age_diagnostics(url)
    google_info = extractor.get_google_index_diagnostics(url)

    for query_info in google_info["queries"]:
        google_query_rows.append({
            "url": url,
            "query": query_info["query"],
            "expected_text": query_info["expected_text"],
            "status_code": query_info["status_code"],
            "has_expected_text": query_info["has_expected_text"],
            "blocked": query_info["blocked"],
            "no_results": query_info["no_results"],
            "result": query_info["result"],
            "error": query_info["error"],
        })

    diagnostic_rows.append({
        "url": url,
        "domain": whois_info["domain"],
        "final_url": final_url,
        "html_ok": html_ok,
        "status_code": status_code,
        "redirects": len(history),
        "whois_ok": whois_info["whois_ok"],
        "domain_age": whois_info["domain_age"],
        "domain_registration_length": whois_info["domain_registration_length"],
        "creation_date_raw": whois_info["creation_date_raw"],
        "expiration_date_raw": whois_info["expiration_date_raw"],
        "creation_date_parsed": whois_info["creation_date_parsed"],
        "expiration_date_parsed": whois_info["expiration_date_parsed"],
        "hostname_resolves": google_info["hostname_resolves"],
        "registered_domain_resolves": google_info["registered_domain_resolves"],
        "google_index": google_info["google_index"],
        "html_error": html_error,
        "whois_error": whois_info["whois_error"],
        "google_error": google_info["error"],
    })

diagnostic_df = pd.DataFrame(diagnostic_rows)
google_query_df = pd.DataFrame(google_query_rows)

display(diagnostic_df)

if google_query_df.empty:
    print("No se ejecutaron consultas a Google Index.")
else:
    display(google_query_df)


,url,domain,final_url,html_ok,status_code,redirects,whois_ok,domain_age,domain_registration_length,creation_date_raw,expiration_date_raw,creation_date_parsed,expiration_date_parsed,hostname_resolves,registered_domain_resolves,google_index,html_error,whois_error,google_error
0,https://www.google.com,google.com,https://www.google.com/,True,200.0,0,True,10526,795,"[datetime.datetime(1997, 9, 15, 4, 0, tzinfo=t...","[datetime.datetime(2028, 9, 14, 4, 0, tzinfo=t...",1997-09-15T04:00:00,2028-09-14T04:00:00,True,True,1,,,
1,https://www.pcfactory.cl,pcfactory.cl,https://www.pcfactory.cl/,True,200.0,0,True,9646,605,2000-02-11 23:21:36+00:00,2028-03-08 20:21:03+00:00,2000-02-11T23:21:36,2028-03-08T20:21:03,True,True,1,,,
2,http://login-banco-seguro.xyz,login-banco-seguro.xyz,http://login-banco-seguro.xyz,False,NaN,0,False,0,0,None,None,None,None,False,False,0,,,


,url,query,expected_text,status_code,has_expected_text,blocked,no_results,result,error
0,https://www.google.com,site:www.google.com,www.google.com,200,True,False,False,True,
1,https://www.pcfactory.cl,site:www.pcfactory.cl,www.pcfactory.cl,200,True,False,False,True,
